<a href="https://colab.research.google.com/github/laubelle/projeto-erva-mate/blob/main/notebooks/regressao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import numpy as np

In [50]:
!git clone https://github.com/laubelle/projeto-erva-mate.git
%cd projeto-erva-mate

Cloning into 'projeto-erva-mate'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 23 (delta 3), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 22.96 KiB | 2.09 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/projeto-erva-mate/projeto-erva-mate


In [51]:
AnaliseSensorialM = pd.read_csv(
    r"data/Análise sensorial.csv",
    sep=',',
    decimal='.',
    encoding='utf-8'
)

CorM = pd.read_csv(
    r"data/Cor.csv",
    sep=',',
    decimal='.',
    encoding='utf-8'
)

EstatUmidM = pd.read_csv(
    r"data/Estat. Umid e AW.csv",
    sep=',',
    decimal='.',
    encoding='utf-8'
)

FenoisIndividuaisM = pd.read_csv(
    r"data/Fenóis individuais.csv",
    sep=';',
    decimal='.',
    encoding='utf-8'
)

In [52]:
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.5f}'.format

In [53]:
def criar_predicao(df, coluna_y):
    modelo = LinearRegression()

    x = df[['Trat_Meses']]
    y = df[coluna_y]

    modelo.fit(x, y)

    # 🔥 intervalo real
    meses_min = df['Trat_Meses'].min()
    meses_max = df['Trat_Meses'].max()

    # 🔵 linha do modelo (mesmo intervalo do histórico)
    meses_linha = pd.DataFrame({
        "Trat_Meses": np.linspace(meses_min, meses_max, 50)
    })

    linha_modelo = modelo.predict(meses_linha)

    df_linha_hist = pd.DataFrame({
        "Trat_Meses": meses_linha["Trat_Meses"],
        coluna_y: linha_modelo,
        "Tipo": "Histórico"
    })

    # 🔴 previsão futura (APÓS o último ponto)
    meses_futuro = pd.DataFrame({
        "Trat_Meses": np.linspace(meses_max, meses_max + 10, 50)
    })

    previsoes = modelo.predict(meses_futuro)

    df_previsao = pd.DataFrame({
        "Trat_Meses": meses_futuro["Trat_Meses"],
        coluna_y: previsoes,
        "Tipo": "Previsão"
    })

    # 🔵 dados reais
    df_real = df.copy()
    df_real["Tipo"] = "Dados Reais"

    return df_real, df_linha_hist, df_previsao, modelo

In [54]:
def gerar_grafico(df_real, df_linha_hist, df_previsao, coluna_y, titulo):

    fig = go.Figure()

    # 🔵 pontos reais
    fig.add_trace(go.Scatter(
        x=df_real["Trat_Meses"],
        y=df_real[coluna_y],
        mode='markers',
        name='Dados Reais'
    ))

    # 🔵 linha do histórico (modelo ajustado)
    fig.add_trace(go.Scatter(
        x=df_linha_hist["Trat_Meses"],
        y=df_linha_hist[coluna_y],
        mode='lines',
        name='Histórico'
    ))

    # 🔴 linha de previsão
    fig.add_trace(go.Scatter(
        x=df_previsao["Trat_Meses"],
        y=df_previsao[coluna_y],
        mode='lines',
        name='Previsão'
    ))

    # 🔥 layout
    fig.update_layout(
        title=titulo,
        xaxis=dict(
            title="Meses",
            tickmode='linear',
            dtick=2
        ),
        yaxis=dict(
            title=coluna_y
        )
    )

    fig.show()


In [55]:
print("Escolha o tipo de análise:")
print("1 - Análise Sensorial")
print("2 - Cor")
print("3 - Fenóis")
print("4 - Estatística de Umidade")

opcao = input("Digite o número: ")

if opcao == "1":
    df = AnaliseSensorialM
    titulo = "Análise Sensorial"

elif opcao == "2":
    df = CorM
    titulo = "Cor"

elif opcao == "3":
    df = FenoisIndividuaisM
    titulo = "Fenóis"

elif opcao == "4":
    df = EstatUmidM
    titulo = "Umidade"

else:
    print("Opção inválida!")
    exit()

#  ESCOLHA DA COLUNA

colunas_validas = [col for col in df.columns if col != "Trat_Meses"]

print("\nEscolha o atributo para análise:")

for i, col in enumerate(colunas_validas):
    print(f"{i} - {col}")

indice = int(input("Digite o número da coluna: "))

if indice < 0 or indice >= len(colunas_validas):
    print("Índice inválido!")
    exit()

coluna_escolhida = colunas_validas[indice]

df_real, df_linha_hist, df_previsao, modelo_regressao = criar_predicao(df, coluna_escolhida)

gerar_grafico(
    df_real,
    df_linha_hist,
    df_previsao,
    coluna_escolhida,
    f"{coluna_escolhida} ao longo dos meses"
)


# Calcula e apresenta R ao quadrado
x = df[['Trat_Meses']]
y = df[coluna_escolhida]
r_squared = modelo_regressao.score(x, y)
print(f"\n📈 R-quadrado do modelo: {r_squared:.4f}")

Escolha o tipo de análise:
1 - Análise Sensorial
2 - Cor
3 - Fenóis
4 - Estatística de Umidade
Digite o número: 1

Escolha o atributo para análise:
0 - Green_color
1 - Fresh_green_odor
2 - Sweet_aroma
3 - Green_aroma
4 - Bitter_aroma
5 - Gold_color
6 - Straw_odor
7 - Straw_aroma
8 - Astringency
Digite o número da coluna: 3



📈 R-quadrado do modelo: 0.8923
